# Детекция ботов по событиям

Задача — дать score каждой куке по её действиям в суточном окне. Метрика: максимальный Precision при Recall не ниже 70%

Этот ноутбук запускает тот же код, что `python solution.py --mode all`. Нужны модули из репозитория, зависимости из `requirements.txt` и исходные данные в `data/`.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from solution import run, sha256, validate_submission

DATA_DIR = Path('data')
OUTPUT_DIR = Path('artifacts')
CONFIG_PATH = Path('final_config.json')
assert CONFIG_PATH.is_file(), 'Откройте ноутбук из корня репозитория.'
print('Python:', sys.version.split()[0])

Python: 3.13.5


## 1. Объекты и временные окна

Одна строка train/test — одна кука. Журнал событий общий для обоих наборов. Сначала проверяем размер и даты

In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['window_start_ts'])
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['window_start_ts'])
pd.DataFrame([
    {'split': 'train', 'cookies': len(train), 'bots': int(train.target.sum()),
     'first_date': train.window_start_ts.min(), 'last_date': train.window_start_ts.max()},
    {'split': 'test', 'cookies': len(test), 'bots': None,
     'first_date': test.window_start_ts.min(), 'last_date': test.window_start_ts.max()},
])

,split,cookies,bots,first_date,last_date
0,train,11091,899.0,2026-04-06,2026-04-19
1,test,4909,NaN,2026-04-20,2026-04-26


## 2. Признаки и модель

События вне `[window_start_ts, window_end_ts)` и полные дубликаты исключаются. Агрегация выполняется отдельно для каждой куки; словари категорий и константные столбцы определяются только на обучающей части.

| Группа | Что описывает | Где смотреть |
|---|---|---|
| Действия и разнообразие | Частоты, контакты, повторы объявлений и запросов | `features.py`: `_add_event_counts`, `_add_diversity` |
| Время и сессии | Концентрацию активности, регулярность пауз, длину сессий | `features.py`: `_add_activity`, `_add_gaps`, `_add_sessions` |
| Навигация | Переходы между действиями, листание выдачи | `_add_transitions`, `_add_search` и функции навигации в `features_extra.py` |
| Курсор | Долю заполненных координат, перемещения и геометрию | `_add_pointer`, `_add_pointer_geometry` |
| Дополнительное время | Короткие паузы и пиковую активность | `_add_trimmed_gaps`, `_add_gap_shape`, `_add_rolling_counts` |

Это группы кандидатов в признаки

Ансамбль: половина веса у среднего трёх CatBoost, половина — у LightGBM на расширенных признаках. Параметры обучения ниже берутся из того же JSON, который читает скрипт.

In [3]:
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
pd.DataFrame([
    {'model': m['family'], 'features': m['features'], 'seeds': m['seeds'],
     'weight': m['weight'], 'parameters': m['params']}
    for m in config['members']
])

,model,features,seeds,weight,parameters
0,catboost,base,"[42, 1337, 2026]",0.5,"{'iterations': 1600, 'depth': 6, 'learning_rat..."
1,lightgbm,augmented,[42],0.5,"{'n_estimators': 300, 'learning_rate': 0.03, '..."


## 3. Повторный запуск

`run(..., 'all')` строит признаки, обучает конфигурацию на датах до 17 апреля и проверяет её на 17–19 апреля. Затем заново обучает четыре модели на всём train и сохраняет ответ для test. Внутри запуска также сравниваются предсказания до и после сохранения моделей.

Файл `submission.csv` в корне - мой эталон. Новый ответ записывается отдельно, в `artifacts/submission.csv`.

In [4]:
run(DATA_DIR, OUTPUT_DIR, CONFIG_PATH, mode='all')

Base features: (16000, 364); clean events: 283833; seconds: 30.02


Extra features: (16000, 329) elapsed 27.38


catboost: seed=42, features=355, seconds=35.3


catboost: seed=1337, features=355, seconds=35.6


catboost: seed=2026, features=355, seconds=35.1


...

catboost: seed=42, features=355, seconds=39.7


catboost: seed=1337, features=355, seconds=38.3


catboost: seed=2026, features=355, seconds=38.1


lightgbm: seed=42, features=665, seconds=4.3


Saved artifacts/submission.csv (4909 rows), SHA-256: 7e85915d437f9ba0795158ab6125f5ff34a800667f6abb5925af665fa4dc2f7e


## 4. Что получилось

Первая таблица сохранённые результаты выбора решения из исходного архива. Вторая - результаты только что выполненной отложенной проверки.

In [5]:
history = pd.read_csv('results/validation_summary.csv')
history.loc[history.model.isin(['ensemble', 'baseline_random_forest']),
            ['fold', 'model', 'n_cookies', 'n_bots', 'precision_at_recall_ge_0_70']]

,fold,model,n_cookies,n_bots,precision_at_recall_ge_0_70
0,early,ensemble,2556,199,0.838323
1,early,baseline_random_forest,2556,199,0.093220
4,late,ensemble,2367,195,0.816568
5,late,baseline_random_forest,2367,195,0.099639
8,holdout,ensemble,1951,160,0.807143
11,holdout,baseline_random_forest,1951,160,0.095718


In [6]:
metrics = json.loads((OUTPUT_DIR / 'holdout_metrics.json').read_text(encoding='utf-8'))
pd.DataFrame(metrics).T[
    ['precision_at_recall_ge_0_70', 'average_precision', 'roc_auc',
     'diagnostic_true_positives', 'diagnostic_false_positives', 'diagnostic_recall']
]

,precision_at_recall_ge_0_70,average_precision,roc_auc,diagnostic_true_positives,diagnostic_false_positives,diagnostic_recall
ensemble,0.807143,0.800142,0.941063,113.0,27.0,0.70625
member_0_catboost,0.784722,0.798470,0.939911,113.0,31.0,0.70625
member_1_lightgbm,0.829630,0.786697,0.937919,112.0,23.0,0.70000
baseline_random_forest,0.095718,0.163416,0.572405,114.0,1077.0,0.71250
constant,0.082009,0.082009,0.500000,160.0,1791.0,1.00000


Одиночный LightGBM лучше ансамбля на отложенной выборке, но хуже на обоих периодах подбора. Он не подставляется вместо ансамбля по результатам этой проверки.

Ограничения: для подбора использовались всего два временных периода; число проверенных вариантов велико; масштаб координат привязан к выданным данным

## 5. Проверка файла и тесты

Совпадение SHA-256 ниже означает совпадение всего CSV, включая порядок строк и запись чисел, а не только равенство метрики. При смене версий библиотек такое совпадение не гарантируется.

In [7]:
new_file = OUTPUT_DIR / 'submission.csv'
submission = pd.read_csv(new_file, float_precision='round_trip')
validate_submission(submission, test)
reference_hash = sha256(Path('submission.csv'))
new_hash = sha256(new_file)
print('rows:', len(submission))
print('reference SHA-256:', reference_hash)
print('new SHA-256:      ', new_hash)
assert new_hash == reference_hash, 'CSV отличается от эталона: проверьте окружение и данные.'
print('CSV полностью совпадает с эталоном.')

rows: 4909
reference SHA-256: 7e85915d437f9ba0795158ab6125f5ff34a800667f6abb5925af665fa4dc2f7e
new SHA-256:       7e85915d437f9ba0795158ab6125f5ff34a800667f6abb5925af665fa4dc2f7e
CSV полностью совпадает с эталоном.


In [8]:
checks = subprocess.run([sys.executable, 'tests.py'], capture_output=True, text=True)
print(checks.stdout + checks.stderr)
checks.check_returncode()

test_boundaries_duplicates_and_grain (__main__.FeatureTests.test_boundaries_duplicates_and_grain) ... ok
test_cookie_identifiers_do_not_affect_features (__main__.FeatureTests.test_cookie_identifiers_do_not_affect_features) ... ok
test_invariance_to_future_events_and_labels (__main__.FeatureTests.test_invariance_to_future_events_and_labels) ... ok
test_against_independent_brute_force (__main__.MetricTests.test_against_independent_brute_force) ... ok
test_official_examples_and_ties (__main__.MetricTests.test_official_examples_and_ties) ... ok
test_reject_invalid_rows_or_scores (__main__.SubmissionTests.test_reject_invalid_rows_or_scores) ... ok
test_valid_format (__main__.SubmissionTests.test_valid_format) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.387s

OK

